In [3]:
from google.colab import files
uploaded = files.upload()

Saving ai4i2020.csv to ai4i2020.csv


In [4]:
import os
print(os.listdir())  # You should see 'ai4i2020.csv' in the list

['.config', 'ai4i2020.csv', 'sample_data']


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded ✅")

Libraries loaded ✅


In [5]:
df = pd.read_csv("ai4i2020.csv")
print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
# UDI and Product ID are just identifiers, not useful for ML
df.drop(columns=['UDI', 'Product ID'], inplace=True)
print("Dropped ID columns ✅")
print(df.columns.tolist())

Dropped ID columns ✅
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


In [7]:
# 'Type' column has L, M, H — convert to numbers
le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])
print("Type encoding:", dict(zip(le.classes_, le.transform(le.classes_))))
df.head()

Type encoding: {'H': np.int64(0), 'L': np.int64(1), 'M': np.int64(2)}


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,2,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,1,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,1,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,1,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [8]:
# Create 2 new meaningful features
df['Power'] = df['Rotational speed [rpm]'] * df['Torque [Nm]']
df['Temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

print("New features added ✅")
print(df[['Power', 'Temp_diff']].describe())

New features added ✅
              Power     Temp_diff
count  10000.000000  10000.000000
mean   59967.147040     10.000630
std    10193.093881      1.001094
min    10966.800000      7.600000
25%    53105.400000      9.300000
50%    59883.900000      9.800000
75%    66873.750000     11.000000
max    99980.400000     12.100000


In [9]:
X = df.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df['Machine failure']

print("Feature shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Feature shape: (10000, 8)
Target distribution:
 Machine failure
0    9661
1     339
Name: count, dtype: int64


In [10]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("Before SMOTE:", y.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_resampled).value_counts().to_dict())

Before SMOTE: {0: 9661, 1: 339}
After SMOTE: {0: 9661, 1: 9661}


In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)

print("Scaling done ✅")
print("Mean (should be ~0):", X_scaled.mean().round(4))

Scaling done ✅
Mean (should be ~0): 0.0


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (15457, 8)
Test size: (3865, 8)


In [13]:
# Save for use in next notebooks
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

X_train_df.to_csv("X_train.csv", index=False)
X_test_df.to_csv("X_test.csv", index=False)
pd.Series(y_train).to_csv("y_train.csv", index=False)
pd.Series(y_test).to_csv("y_test.csv", index=False)

print("Processed data saved ✅")

Processed data saved ✅
